# LAVKA Incremental Loader

Инкрементальная загрузка данных Лавки из SharePoint в `ECOM_ETL`.  
Трекинг загруженных файлов — через `_source_file` в целевых таблицах (без реестра и VIEW).

In [ ]:
import os
import re
from datetime import datetime

import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DateType

notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
team_folder = notebook_path.split("/")[2]

import sys
sys.path.append(f"/Workspace/eperfectstore-prod/{team_folder}/notebooks/eperfectstore-prod/e-com/COMMON_FUNCTIONS_AND_CONSTANTS_FOLDER/")
from common_functions_and_constants import *

RUN_TS = datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%SZ")
print(f"Run started: {RUN_TS}")

In [ ]:
PO1_URL = "https://pepsico.sharepoint.com/teams/RussiaSPO1CustomerCollaboration/"
WBD_URL = "https://pepsico.sharepoint.com/teams/AzureCCplatform/"

PATHS = {
    "po1_main":    {"spo": "Shared Documents/General/E-COM/Клиенты/2.Лавка/DOS отчет/DOS REP/Метрики cpfr/",        "dbfs": "/dbfs/ecom/lavka_dos_po1_test",                "scan": "/ecom/lavka_dos_po1_test"},
    "po1_archive": {"spo": "Shared Documents/General/E-COM/Клиенты/2.Лавка/DOS отчет/DOS REP/Метрики cpfr/archiv",  "dbfs": "/dbfs/ecom/lavka_dos_po1_test_archive",        "scan": "/ecom/lavka_dos_po1_test_archive"},
    "po1_kub":     {"spo": "Shared Documents/General/E-COM/Клиенты/2.Лавка/Для КУБ Лавка/",                         "dbfs": "/dbfs/ecom/lavka_dos_po1_kub",                 "scan": "/ecom/lavka_dos_po1_kub"},
    "po1_dicts":   {"spo": "Shared Documents/General/E-COM/Клиенты/2.Лавка/DICTIONARIES/",                          "dbfs": "/dbfs/ecom/lavka_dos_po1_test/DICTIONARIES"},
    "wbd_ao":      {"spo": "Shared Documents/WBD/Ecom/Lavka/CPFR_metrics/metrics_ao/",                              "dbfs": "/dbfs/ecom/lavka_dos_wbd/",                    "scan": "/ecom/lavka_dos_wbd/"},
    "wbd_orders":  {"spo": "Shared Documents/WBD/Ecom/Lavka/CPFR_metrics/metrics_orders/",                          "dbfs": "/dbfs/ecom/lavka_dos_wbd/metrics_orders/",     "scan": "/ecom/lavka_dos_wbd/metrics_orders/"},
    "wbd_osa":     {"spo": "Shared Documents/WBD/Ecom/Lavka/CPFR_metrics/metrics_osa/",                             "dbfs": "/dbfs/ecom/lavka_dos_wbd/metrics__osa/",       "scan": "/ecom/lavka_dos_wbd/metrics__osa/"},
    "wbd_pred":    {"spo": "Shared Documents/WBD/Ecom/Lavka/CPFR_metrics/metrics_prediction/",                      "dbfs": "/dbfs/ecom/lavka_dos_wbd/metrics_prediction/", "scan": "/ecom/lavka_dos_wbd/metrics_prediction/"},
    "wbd_ss":      {"spo": "Shared Documents/WBD/Ecom/Lavka/CPFR_metrics/metrics_sales_stock/",                     "dbfs": "/dbfs/ecom/lavka_dos_wbd/metrics_sales_stock/","scan": "/ecom/lavka_dos_wbd/metrics_sales_stock/"},
    "wbd_dir":     {"spo": "Shared Documents/WBD/Ecom/Lavka/CPFR_metrics/directory/",                               "dbfs": "/dbfs/ecom/lavka_dos_wbd/directory/"},
}

BASE = "abfss://eperfectstore@pepedapprdlakesukeu01.dfs.core.windows.net/data/silver/lavka"
TARGETS = {
    "po1":        {"table": "ECOM_ETL.LAVKA_PO1",                     "path": f"{BASE}/lavka_po1"},
    "po1_assort": {"table": "ECOM_ETL.LAVKA_PO1_ASSORT",              "path": f"{BASE}/lavka_po1_assort"},
    "wbd_ao":     {"table": "ECOM_ETL.LAVKA_WBD_METRICS_AO",          "path": f"{BASE}/lavka_wbd/metrics_ao"},
    "wbd_orders": {"table": "ECOM_ETL.LAVKA_WBD_METRICS_ORDERS",      "path": f"{BASE}/lavka_wbd/metrics_orders"},
    "wbd_osa":    {"table": "ECOM_ETL.LAVKA_WBD_METRICS_OSA",         "path": f"{BASE}/lavka_wbd/metrics_osa"},
    "wbd_pred":   {"table": "ECOM_ETL.LAVKA_WBD_METRICS_PREDICTION",  "path": f"{BASE}/lavka_wbd/metrics_prediction"},
    "wbd_ss":     {"table": "ECOM_ETL.LAVKA_WBD_METRICS_SALES_STOCK", "path": f"{BASE}/lavka_wbd/metrics_sales_stock"},
    "wbd_dir":    {"table": "ECOM_ETL.LAVKA_WBD_DIRECTORY",           "path": f"{BASE}/lavka_wbd/directory"},
}

# автоматически — все scan-директории из PATHS
CLEANUP_DIRS = [p["scan"] for p in PATHS.values() if "scan" in p]

In [ ]:
def log(msg):
    print(f"[{datetime.utcnow().strftime('%Y-%m-%d %H:%M:%S')} UTC] {msg}")


def normalize_spark_columns(df):
    for c in df.columns:
        new_c = re.sub(r"_+", "_", re.sub(r"[^0-9a-zA-Z]+", "_", c.strip().lower())).strip("_")
        if new_c != c:
            df = df.withColumnRenamed(c, new_c)
    return df


def parse_metric_date(file_path):
    m = re.search(r"\d{8}", os.path.splitext(os.path.basename(file_path))[0])
    if not m:
        return None
    for fmt in ("%d%m%Y", "%Y%m%d"):
        try:
            return datetime.strptime(m.group(), fmt).date()
        except ValueError:
            pass
    return None


def table_exists(table_name):
    try:
        spark.read.table(table_name).limit(1).count()
        return True
    except Exception:
        return False


def list_new_files(scan_dir, ext, target_table):
    try:
        all_files = [
            (f.path.replace("dbfs:/", "/dbfs/"), os.path.basename(f.path), parse_metric_date(f.path))
            for f in dbutils.fs.ls(scan_dir) if f.path.lower().endswith(ext.lower())
        ]
    except Exception as e:
        log(f"ls failed {scan_dir}: {e}")
        return []

    if not all_files:
        return []

    schema = StructType([
        StructField("full_path", StringType(), False),
        StructField("_source_file", StringType(), False),
        StructField("_file_date", DateType(), True),
    ])
    files_df = spark.createDataFrame(all_files, schema=schema)

    if not table_exists(target_table):
        return [r[0] for r in files_df.select("full_path").collect()]

    target_df = spark.read.table(target_table)
    if "_source_file" in target_df.columns:
        new_df = files_df.join(target_df.select("_source_file").distinct(), "_source_file", "left_anti")
        return [r[0] for r in new_df.select("full_path").collect()]

    if "metric_date" in target_df.columns:
        max_date = target_df.select(F.max("metric_date")).collect()[0][0]
        if max_date:
            return [r[0] for r in files_df.filter(F.col("_file_date") > F.lit(max_date)).select("full_path").collect()]

    return [r[0] for r in files_df.select("full_path").collect()]


def read_po1_excels(paths):
    """
    Читает PO1-файлы. Пробует листы по порядку: "Лист1" → "Sheet1" → первый лист (sheet_name=0).
    Логирует, какие файлы пропускаются и почему.
    """
    required = ["Date", "City", "Supplier", "Item", "Item_Name", "Metric", "Value"]
    dfs = []
    for path in paths:
        fname = os.path.basename(path)
        loaded = False
        for sheet in ["Лист1", "Sheet1", 0]:
            try:
                df = pd.read_excel(path, sheet_name=sheet, header=0)
            except Exception:
                continue
            missing = set(required) - set(df.columns)
            if missing:
                log(f"SKIP {fname} sheet={sheet!r}: отсутствуют колонки {sorted(missing)}, найдены {df.columns.tolist()}")
                continue
            cols = required + (["w"] if "w" in df.columns else [])
            df = df[cols].astype({c: "string" for c in ["City", "Supplier", "Item", "Item_Name", "Metric", "Value"]})
            df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
            if "w" in df.columns:
                df["w"] = pd.to_numeric(df["w"], errors="coerce").astype("Int64")
            df["metric_date"] = pd.to_datetime(parse_metric_date(path))
            df["_source_file"] = fname
            dfs.append(df)
            loaded = True
            break
        if not loaded:
            log(f"SKIP {fname}: ни один лист не подошёл")
    if not dfs:
        return None
    return normalize_spark_columns(spark.createDataFrame(pd.concat(dfs, ignore_index=True)))


def read_csv_batch(paths):
    dfs = []
    for path in paths:
        try:
            df = pd.read_csv(path, header=0)
            df["_source_file"] = os.path.basename(path)
            dfs.append(df)
        except Exception as e:
            log(f"SKIP CSV {os.path.basename(path)}: {e}")
    if not dfs:
        return None
    return normalize_spark_columns(spark.createDataFrame(pd.concat(dfs, ignore_index=True)))


def append_to_table(df, target):
    (
        df.withColumn("_loaded_at", F.current_timestamp())
        .write.mode("append")
        .option("mergeSchema", "true")
        .option("path", target["path"])
        .saveAsTable(target["table"])
    )
    log(f"{target['table']}: appended")

## 1) PO1 — Main + Archive + КУБ → LAVKA_PO1

In [ ]:
for key in ["po1_main", "po1_archive", "po1_kub"]:
    copy_from_spo(PO1_URL, PATHS[key]["spo"], PATHS[key]["dbfs"])

all_po1_files = sum(
    [list_new_files(PATHS[key]["scan"], ".xlsx", TARGETS["po1"]["table"]) for key in ["po1_main", "po1_archive", "po1_kub"]],
    []
)
log(f"PO1 новых файлов: {len(all_po1_files)}")

batch = read_po1_excels(all_po1_files)
if batch is not None:
    append_to_table(batch, TARGETS["po1"])
else:
    log("PO1: нечего загружать")

## 2) PO1 Dictionaries → LAVKA_PO1_ASSORT

In [ ]:
copy_from_spo(PO1_URL, PATHS["po1_dicts"]["spo"], PATHS["po1_dicts"]["dbfs"])

assort_df = normalize_spark_columns(
    spark.createDataFrame(pd.read_excel(f"{PATHS['po1_dicts']['dbfs']}/Assort Lavka.xlsx", sheet_name="Assort"))
)
assort_df.write.mode("overwrite").option("path", TARGETS["po1_assort"]["path"]).saveAsTable(TARGETS["po1_assort"]["table"])
log(f"{TARGETS['po1_assort']['table']}: overwritten")

## 3) WBD Metrics (инкрементально)

In [ ]:
for key in ["wbd_ao", "wbd_orders", "wbd_osa", "wbd_pred", "wbd_ss"]:
    p, t = PATHS[key], TARGETS[key]
    copy_from_spo(WBD_URL, p["spo"], p["dbfs"])

    new_files = list_new_files(p["scan"], ".csv", t["table"])
    log(f"{t['table']}: {len(new_files)} новых файлов")
    if not new_files:
        continue

    batch = read_csv_batch(new_files)
    if batch is None:
        continue

    if "date" in batch.columns:
        batch = batch.withColumn("date", F.to_date("date"))
    if "metrics_version" in batch.columns:
        batch = batch.withColumn("metrics_version", F.expr("try_cast(metrics_version as date)"))
        batch = batch.filter(F.col("metrics_version").isNotNull())

    append_to_table(batch, t)

## 4) WBD Directory → LAVKA_WBD_DIRECTORY

In [ ]:
copy_from_spo(WBD_URL, PATHS["wbd_dir"]["spo"], PATHS["wbd_dir"]["dbfs"])

products_df = normalize_spark_columns(spark.createDataFrame(pd.read_csv(f"{PATHS['wbd_dir']['dbfs']}/products.csv")))
products_df.write.mode("overwrite").option("overwriteSchema", "True").option("path", TARGETS["wbd_dir"]["path"]).saveAsTable(TARGETS["wbd_dir"]["table"])
log(f"{TARGETS['wbd_dir']['table']}: overwritten")

## 5) Cleanup staging

In [ ]:
for d in CLEANUP_DIRS:
    try:
        for f in dbutils.fs.ls(d):
            dbutils.fs.rm(f.path, True)
        log(f"Cleaned: {d}")
    except Exception as e:
        log(f"Skip {d}: {e}")

log("Run finished")